# Ανάλυση αποτελεσμάτων

Το notebook διαβάζει τα αποθηκευμένα classification reports των μοντέλων
και υπολογίζει τις διαφορετικές εκδοχές του Macro-F1.


In [ ]:
from pathlib import Path

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns


RESULTS_DIR = Path("results")
MANIFEST_PATH = Path("data/zoolake_clean_split_manifest.csv")

In [ ]:
manifest = pd.read_csv(MANIFEST_PATH)

# Μετράμε πόσες εικόνες έχει συνολικά κάθε κλάση.
class_counts = manifest["label"].value_counts()

class_counts_table = class_counts.to_frame(
    name="number_of_images"
)

display(class_counts_table)

In [ ]:
# Όλες οι κλάσεις του dataset.
all_classes = class_counts.index.tolist()

# Κλάσεις με τουλάχιστον 200 εικόνες συνολικά.
classes_ge_200 = class_counts[
    class_counts >= 200
].index.tolist()

# Κατηγορίες που δεν αντιστοιχούν σε συγκεκριμένο οργανισμό.
excluded_classes = [
    "dirt",
    "unknown",
    "unknown_plankton"
]

classes_no_excluded = [
    class_name
    for class_name in all_classes
    if class_name not in excluded_classes
]

classes_ge_200_no_excluded = [
    class_name
    for class_name in classes_ge_200
    if class_name not in excluded_classes
]


print("Όλες οι κλάσεις:", len(all_classes))
print("Χωρίς τις 3 γενικές κατηγορίες:", len(classes_no_excluded))
print("Με τουλάχιστον 200 εικόνες:", len(classes_ge_200))
print(
    "Με τουλάχιστον 200 εικόνες χωρίς τις γενικές κατηγορίες:",
    len(classes_ge_200_no_excluded)
)

In [ ]:
report_paths = {
    "Custom CNN": RESULTS_DIR
    / "custom_cnn"
    / "custom_cnn_clean_split_lanczos_seed12345_results"
    / "classification_report.csv",

    "Custom CNN + oversampling": RESULTS_DIR
    / "custom_cnn"
    / "custom_cnn_clean_split_lanczos_oversampling_to100_seed12345_results"
    / "classification_report.csv",

    "EfficientNetB2 frozen": RESULTS_DIR
    / "efficientnetb2"
    / "efficientnetb2_frozen_backbone"
    / "classification_report.csv",

    "EfficientNetB2 BN-locked": RESULTS_DIR
    / "efficientnetb2"
    / "efficientnetb2_bn_locked_results"
    / "classification_report.csv",

    "EfficientNetB2 BN-locked + oversampling": RESULTS_DIR
    / "efficientnetb2"
    / "efficientnetb2_bn_locked_oversampling_results"
    / "classification_report.csv",

    "EfficientNetB2 full unfreeze": RESULTS_DIR
    / "efficientnetb2"
    / "efficientnetb2_clean_split_full_unfreeze_results"
    / "classification_report.csv",

    "EfficientNetB2 from scratch": RESULTS_DIR
    / "efficientnetb2"
    / "efficientnetb2_clean_split_from_scratch_seed12345_results"
    / "classification_report.csv",

    "MobileNetV1 frozen": RESULTS_DIR
    / "mobilenet"
    / "mobilenet_frozen_backbone_results"
    / "classification_report.csv",

    "MobileNetV1 BN-locked": RESULTS_DIR
    / "mobilenet"
    / "mobilenet_clean_split_bn_locked_results"
    / "classification_report.csv",

    "MobileNetV1 BN-locked + oversampling": RESULTS_DIR
    / "mobilenet"
    / "mobilenet_bn_locked_oversampling_results"
    / "classification_report.csv",

    "MobileNetV1 full unfreeze": RESULTS_DIR
    / "mobilenet"
    / "mobilenet_clean_split_full_unfreeze_results"
    / "classification_report.csv",

    "MobileNetV1 from scratch": RESULTS_DIR
    / "mobilenet"
    / "mobilenet_clean_split_from_scratch_seed12345_results"
    / "classification_report.csv",

    "ConvNeXtBase": RESULTS_DIR
    / "convnextbase"
    / "full_finetune_test_classification_report.csv"
}

In [ ]:
reports = {}

for model_name, report_path in report_paths.items():
    report = pd.read_csv(
        report_path,
        index_col=0
    )

    # Στο report του ConvNeXt η κλάση είχε την παλιά ορθογραφία.
    report = report.rename(
        index={"kellikottia": "kellicottia"}
    )

    reports[model_name] = report

print("Διαβάστηκαν reports για", len(reports), "μοντέλα.")

In [ ]:
rows = []

for model_name, report in reports.items():

    f1_scores = report["f1-score"]

    rows.append({
        "model": model_name,

        "accuracy": report.loc[
            "accuracy",
            "f1-score"
        ],

        "macro_f1_all": f1_scores.loc[
            all_classes
        ].mean(),

        "macro_f1_no_excluded": f1_scores.loc[
            classes_no_excluded
        ].mean(),

        "macro_f1_ge_200": f1_scores.loc[
            classes_ge_200
        ].mean(),

        "macro_f1_ge_200_no_excluded": f1_scores.loc[
            classes_ge_200_no_excluded
        ].mean()
    })


results_table = pd.DataFrame(rows)

results_table = results_table.set_index("model")

# Μετατρέπουμε τα αποτελέσματα σε ποσοστά.
results_table = results_table * 100

display(results_table.round(2))

In [ ]:
frozen_report = reports[
    "MobileNetV1 frozen"
]

bn_locked_report = reports[
    "MobileNetV1 BN-locked"
]


mobile_comparison = pd.DataFrame({
    "frozen_f1": frozen_report.loc[
        all_classes,
        "f1-score"
    ],

    "bn_locked_f1": bn_locked_report.loc[
        all_classes,
        "f1-score"
    ]
})


mobile_comparison["difference"] = (
    mobile_comparison["bn_locked_f1"]
    - mobile_comparison["frozen_f1"]
).round(10)


display(mobile_comparison)

In [ ]:
bn_better = (
    mobile_comparison["difference"] > 0
).sum()

same = (
    mobile_comparison["difference"] == 0
).sum()

frozen_better = (
    mobile_comparison["difference"] < 0
).sum()


print("Καλύτερο το BN-locked:", bn_better)
print("Ίδιο F1:", same)
print("Καλύτερο το frozen:", frozen_better)

In [ ]:
display(
    mobile_comparison.loc[["chaoborus"]]
)

test_images = frozen_report.loc[
    "chaoborus",
    "support"
]

effect = (
    mobile_comparison.loc[
        "chaoborus",
        "frozen_f1"
    ]
    - mobile_comparison.loc[
        "chaoborus",
        "bn_locked_f1"
    ]
) / len(all_classes)


print("Εικόνες chaoborus στο test:", int(test_images))
print(
    "Επίδραση στο Macro-F1:",
    round(effect * 100, 2),
    "ποσοστιαίες μονάδες"
)

In [ ]:
classes_without_chaoborus = [
    class_name
    for class_name in all_classes
    if class_name != "chaoborus"
]


frozen_without_chaoborus = frozen_report.loc[
    classes_without_chaoborus,
    "f1-score"
].mean()

bn_locked_without_chaoborus = bn_locked_report.loc[
    classes_without_chaoborus,
    "f1-score"
].mean()


print(
    "Frozen χωρίς chaoborus:",
    round(frozen_without_chaoborus * 100, 2),
    "%"
)

print(
    "BN-locked χωρίς chaoborus:",
    round(bn_locked_without_chaoborus * 100, 2),
    "%"
)

In [ ]:
frozen_predictions = pd.read_csv(
    RESULTS_DIR
    / "mobilenet"
    / "mobilenet_frozen_backbone_results"
    / "test_predictions.csv"
)

bn_locked_predictions = pd.read_csv(
    RESULTS_DIR
    / "mobilenet"
    / "mobilenet_clean_split_bn_locked_results"
    / "test_predictions.csv"
)


print("Frozen:")
display(
    frozen_predictions[
        frozen_predictions["true_label"] == "chaoborus"
    ][
        ["true_label", "predicted_label", "confidence"]
    ]
)


print("BN-locked:")
display(
    bn_locked_predictions[
        bn_locked_predictions["true_label"] == "chaoborus"
    ][
        ["true_label", "predicted_label", "confidence"]
    ]
)